# Qwen3 Local Generation Test Notebook

Use this notebook to load the local Qwen checkpoint, verify that the checkpoint dimensions match the custom model, and generate answers with the correct Qwen chat prompt format.


## 1. Setup

Run this once after opening the notebook. It makes the repo root importable and sets the local model directory.


In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

MODEL_DIR = Path("qwen")
MODEL_DIR.resolve()


## 2. Check Downloaded Files

The model directory should contain `config.json`, `model.safetensors`, `tokenizer.json`, and `tokenizer_config.json`.


In [ ]:
required_files = [
    "config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
]

missing = [name for name in required_files if not (MODEL_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing files in {MODEL_DIR.resolve()}: {missing}")

for name in required_files:
    p = MODEL_DIR / name
    print(f"{name}: {p.stat().st_size:,} bytes")


## 3. Read Config And Verify Weight Shapes

Qwen3-0.6B has `hidden_size=1024`, `num_attention_heads=16`, and `head_dim=128`, so the query/output attention projection dimensions use `16 * 128 = 2048`. That is expected.


In [ ]:
import json
from safetensors import safe_open

with open(MODEL_DIR / "config.json", "r", encoding="utf-8") as f:
    hf_config = json.load(f)

print("model_type:", hf_config["model_type"])
print("hidden_size:", hf_config["hidden_size"])
print("num_hidden_layers:", hf_config["num_hidden_layers"])
print("num_attention_heads:", hf_config["num_attention_heads"])
print("num_key_value_heads:", hf_config["num_key_value_heads"])
print("head_dim:", hf_config["head_dim"])
print("vocab_size:", hf_config["vocab_size"])

with safe_open(MODEL_DIR / "model.safetensors", framework="pt", device="cpu") as f:
    shape_checks = {
        "model.embed_tokens.weight": (hf_config["vocab_size"], hf_config["hidden_size"]),
        "model.layers.0.self_attn.q_proj.weight": (hf_config["num_attention_heads"] * hf_config["head_dim"], hf_config["hidden_size"]),
        "model.layers.0.self_attn.k_proj.weight": (hf_config["num_key_value_heads"] * hf_config["head_dim"], hf_config["hidden_size"]),
        "model.layers.0.self_attn.v_proj.weight": (hf_config["num_key_value_heads"] * hf_config["head_dim"], hf_config["hidden_size"]),
        "model.layers.0.self_attn.o_proj.weight": (hf_config["hidden_size"], hf_config["num_attention_heads"] * hf_config["head_dim"]),
        "model.layers.0.mlp.gate_proj.weight": (hf_config["intermediate_size"], hf_config["hidden_size"]),
        "model.layers.0.mlp.up_proj.weight": (hf_config["intermediate_size"], hf_config["hidden_size"]),
        "model.layers.0.mlp.down_proj.weight": (hf_config["hidden_size"], hf_config["intermediate_size"]),
        "lm_head.weight": (hf_config["vocab_size"], hf_config["hidden_size"]),
    }

    for key, expected_shape in shape_checks.items():
        actual_shape = tuple(f.get_tensor(key).shape)
        print(f"{key}: {actual_shape}")
        assert actual_shape == expected_shape, f"{key}: expected {expected_shape}, got {actual_shape}"

print("All checked dimensions match the config.")


## 4. Load Tokenizer

For chat-style answers, do not encode the raw prompt directly. Use `encode_chat(...)` or `generate_text(..., chat=True)` so the prompt includes Qwen's `<|im_start|>` / `<|im_end|>` markers and assistant generation prompt.


In [ ]:
from base_model.qwen import Qwen3Tokenizer

tokenizer = Qwen3Tokenizer(MODEL_DIR / "tokenizer.json")

print("EOS:", tokenizer.eos_token, tokenizer.eos_token_id)
print("PAD:", tokenizer.pad_token, tokenizer.pad_token_id)
print("Stop IDs:", sorted(tokenizer.stop_token_ids))

preview_ids = tokenizer.encode_chat(
    "Explain large language models.",
    add_generation_prompt=True,
    enable_thinking=False,
)
print(tokenizer.decode(preview_ids))


## 5. Load Model

This loads the local safetensors checkpoint into the custom Qwen implementation. On a CUDA machine this should run quickly; CPU will be much slower.


In [ ]:
import torch
from safetensors.torch import load_file

from base_model.qwen import (
    QWEN_CONFIG_06_B,
    Qwen3Model,
    generate_text,
    load_hf_weights_into_qwen,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

weights = load_file(MODEL_DIR / "model.safetensors")
model = Qwen3Model(QWEN_CONFIG_06_B)
load_hf_weights_into_qwen(
    model,
    param_config={
        "n_layers": QWEN_CONFIG_06_B["n_layers"],
        "hidden_dim": QWEN_CONFIG_06_B["hidden_dim"],
    },
    params=weights,
)

model.to(device)
model.to(torch.bfloat16)
model.eval()

print(f"Loaded {sum(p.numel() for p in model.parameters()):,} parameters")


## 6. Generate An Answer

This is the main cell to edit while testing. It uses the chat template and generates many tokens, not just one next token.


In [ ]:
prompt = "Explain large language models in two sentences."

response = generate_text(
    model,
    tokenizer,
    prompt,
    max_new_tokens=128,
    temperature=0,
    chat=True,
    enable_thinking=False,
)

print(response)


## 7. Inspect The First Next Token

This is only for debugging logits. A single next token is not a full answer.


In [ ]:
input_ids = tokenizer.encode_chat(
    prompt,
    add_generation_prompt=True,
    enable_thinking=False,
)
input_tensor = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)

with torch.inference_mode():
    logits = model(input_tensor)

last_logits = logits[0, -1].float()
probs = torch.softmax(last_logits, dim=-1)
top_probs, top_ids = torch.topk(probs, k=10)

for prob, token_id in zip(top_probs.tolist(), top_ids.tolist()):
    print(f"{token_id:>6}  {prob:.4%}  {tokenizer.decode([token_id])!r}")
